# NT-v2-50M on splice sites: frozen vs fine-tuned

Same sample, same head, two runs:

* **Run A, frozen**: the backbone is not trained. Centre token + previous token, concatenated, into a small head. This is the local pipeline's `center_plus_prev_concat` setup (0.9405 test accuracy in `results/phase1_results.json`).
* **Run B, fine-tuned**: the same head, but the whole backbone is trained too.

Self-contained: everything it needs from the local repo (`02_dna-splicing-pipeline/scripts/`) is copied in below, and each piece says which file it came from.

**Kaggle settings:** Accelerator = GPU T4, Internet = On. Then *Run All*. Results land in `results.json` in the working directory.

In [ ]:
# Same pin as dna-venv (scripts/setup_local_env.sh). Must run before anything imports transformers.
%pip install -q transformers==4.49.0 einops

In [ ]:
SMOKE = True   # True = 200 train / 100 test / 1 epoch (does it run?). False = the full run.

MODEL_REPO = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
MODEL_REVISION = "81b29e5786726d891dbf929404ef20adca5b36f1"     # the snapshot dna-venv has cached
DATASET_REPO = "InstaDeepAI/nucleotide_transformer_downstream_tasks_revised"   # phase1_common.py
DATASET_CONFIG = "splice_sites_all"                                            # phase1_common.py
DATASET_REVISION = "851f9946252e90c665cdb3cc3eedb78f1f26197c"   # data/label_info.json: dataset_commit

SEED = 42                # sampling seed (phase1_common.SEED)
HEAD_SEED = 0            # 03_train_head.py --seed: validation split and head initialisation
N_TRAIN, N_TEST = (200, 100) if SMOKE else (8000, 2000)   # 02b_embed_center.py --max-train / --max-test
VAL_FRAC = 0.10          # 03_train_head.py --val-frac
CENTER_BASE = 300        # 02b_embed_center.py / 12_repool_center.py

# Run B (fine-tune)
FT_EPOCHS = 1 if SMOKE else 2
BATCH_SIZE = 16
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 2e-5, 1e-3, 0.01

# Run A head recipe = 03_train_head.py defaults, the recipe behind the local 0.9405.
# The head trains on cached vectors, so this costs seconds and is the same in SMOKE mode.
A_LR, A_WEIGHT_DECAY, A_BATCH, A_MAX_EPOCHS, A_PATIENCE = 1e-3, 0.0, 128, 300, 20

# data/label_info.json (worked out from the motifs by 01_get_data.py; re-checked below)
LABEL_NAMES = {0: "none", 1: "acceptor", 2: "donor"}
MOTIF_OFFSETS = {"donor_GT_plus": 301, "donor_AC_minus": 298,
                 "acceptor_AG_plus": 298, "acceptor_CT_minus": 301}
TRAIN_CHROMS = {str(i) for i in range(1, 20)} | {"X", "Y"}
TEST_CHROMS = {"20", "21"}
# center_meta.json run_key: fingerprints of the exact 8000 / 2000 rows the local run used
LOCAL_SAMPLE_SHA1 = {"train": "d5b52e4e9707cde5", "test": "181e6bfb2b6dcc65"}

In [ ]:
import collections, copy, hashlib, json, platform, time

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from torch import nn

T_START = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # one GPU, no DataParallel
USE_AMP = DEVICE.type == "cuda"      # fp16 autocast on GPU only. Set False here if the loss goes NaN.
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None

print(f"python {platform.python_version()}  torch {torch.__version__}  "
      f"transformers {transformers.__version__}  sklearn {sklearn.__version__}")
print(f"device: {DEVICE}" + (f" ({GPU_NAME})" if GPU_NAME else "") + f"   fp16 autocast: {USE_AMP}")
print(f"SMOKE = {SMOKE}: {N_TRAIN} train / {N_TEST} test, {FT_EPOCHS} fine-tune epoch(s)")
assert transformers.__version__ == "4.49.0", "restart the kernel so the pinned transformers is the one imported"
if not SMOKE and DEVICE.type != "cuda":
    print("WARNING: full run on CPU will take hours. On Kaggle: Settings -> Accelerator -> GPU T4.")

## Data: same parquet files, same chromosome split, same seeded sample

In [ ]:
def load_splits():
    # 01_get_data.py download(): the repo has no named config for this task, so the
    # task's two parquet files are loaded directly. Pinned to the commit the local run used.
    files = {s: f"{DATASET_CONFIG}/{s}.parquet" for s in ("train", "test")}
    try:
        from datasets import load_dataset
        ds = load_dataset(DATASET_REPO, data_files=files, revision=DATASET_REVISION)
        return {s: ds[s].to_pandas() for s in files}
    except Exception as exc:        # `datasets` missing or broken by the pin: read the same files directly
        print(f"(datasets.load_dataset failed: {type(exc).__name__}: {str(exc)[:200]}; reading parquet with pandas)")
        from huggingface_hub import hf_hub_download
        return {s: pd.read_parquet(hf_hub_download(DATASET_REPO, f, repo_type="dataset",
                                                   revision=DATASET_REVISION))
                for s, f in files.items()}


def stratified_subsample(labels, n, seed=SEED):
    # phase1_common.py stratified_subsample(), unchanged
    labels = np.asarray(labels)
    total = len(labels)
    if n is None or n < 0 or n >= total:
        return np.arange(total)
    from sklearn.model_selection import train_test_split

    idx, _ = train_test_split(np.arange(total), train_size=n,
                              stratify=labels, random_state=seed)
    return np.sort(idx)


def indices_digest(idx):
    # 02_embed.py indices_digest(), unchanged
    return hashlib.sha1(np.ascontiguousarray(idx, dtype=np.int64).tobytes()).hexdigest()[:16]


full = load_splits()

# Chromosome split. The dataset's own train/test files ARE the split; this checks it,
# the way phase1_common.py dataset_chromosomes() reads it from the `name` column.
chroms = {s: {n.split(":")[0].removeprefix("chr") for n in full[s]["name"]} for s in full}
assert chroms["train"] <= TRAIN_CHROMS, f"unexpected train chromosomes: {sorted(chroms['train'] - TRAIN_CHROMS)}"
assert chroms["test"] <= TEST_CHROMS, f"unexpected test chromosomes: {sorted(chroms['test'] - TEST_CHROMS)}"
assert not (chroms["train"] & chroms["test"])
print(f"train: {len(full['train'])} rows on {len(chroms['train'])} chromosomes (chr1-19, X, Y)   "
      f"test: {len(full['test'])} rows on chr{', chr'.join(sorted(chroms['test']))}")

# Seeded class-stratified sample, as in 02b_embed_center.py main()
idx = {"train": stratified_subsample(full["train"]["label"], N_TRAIN, SEED),
       "test": stratified_subsample(full["test"]["label"], N_TEST, SEED)}
seqs = {s: full[s]["sequence"].iloc[idx[s]].tolist() for s in idx}
y = {s: full[s]["label"].to_numpy()[idx[s]].astype(np.int64) for s in idx}
CLASSES = sorted(LABEL_NAMES)
assert all(set(y[s].tolist()) <= set(CLASSES) for s in y)

SAME_SAMPLE_AS_LOCAL = None
if not SMOKE:
    SAME_SAMPLE_AS_LOCAL = all(indices_digest(idx[s]) == LOCAL_SAMPLE_SHA1[s] for s in idx)
    print("sample identical to the local 8000 / 2000 rows:", SAME_SAMPLE_AS_LOCAL)
    if not SAME_SAMPLE_AS_LOCAL:
        print("WARNING: this sklearn drew different rows than the local run. Run A and Run B still "
              "share one sample, but Run A is no longer row-for-row the local 0.9405 run.")

# 10% validation rows carved from train: the split 03_train_head.py makes
# (train_test_split(test_size=0.10, stratify=y, random_state=seed)). Both runs use it.
from sklearn.model_selection import train_test_split
fit_rows, val_rows = train_test_split(np.arange(len(y["train"])), test_size=VAL_FRAC,
                                      stratify=y["train"], random_state=HEAD_SEED)
for s in idx:
    print(f"{s:>5}: {len(idx[s])} rows, per class "
          f"{ {LABEL_NAMES[c]: int((y[s] == c).sum()) for c in CLASSES} }")
print(f"train -> {len(fit_rows)} fit + {len(val_rows)} validation")

# Label names re-checked the way 01_get_data.py found them: donors carry GT at offset 301
# (or AC at 298 on the other strand), acceptors AG at 298 (or CT at 301).
def has(seq_list, a, ka, b, kb):
    return float(np.mean([s[ka:ka + 2] == a or s[kb:kb + 2] == b for s in seq_list]))

o = MOTIF_OFFSETS
for c in CLASSES:
    rows = [s for s, lab in zip(seqs["train"], y["train"]) if lab == c]
    d = has(rows, "GT", o["donor_GT_plus"], "AC", o["donor_AC_minus"])
    a = has(rows, "AG", o["acceptor_AG_plus"], "CT", o["acceptor_CT_minus"])
    print(f"label {c} ({LABEL_NAMES[c]:>8}): donor motif {d:.1%}   acceptor motif {a:.1%}")
    if LABEL_NAMES[c] == "donor":
        assert d > 0.9 and a < 0.3, "label 2 does not look like donor"
    if LABEL_NAMES[c] == "acceptor":
        assert a > 0.9 and d < 0.3, "label 1 does not look like acceptor"

## Tokens: which input position is the centre?

NT-v2 cuts the sequence into 6-letter tokens and puts a `<cls>` token in front. The index is not hard-coded: every sequence's tokens are mapped back to the bases they cover, and the token covering base 300 is looked up. A wrong index would not crash, it would just quietly read the wrong place, so this cell asserts everything and prints what it found.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION, trust_remote_code=True)
special_ids = np.array(sorted(set(tok.all_special_ids)))
max_len = int(min(tok.model_max_length, 2048))


def tokenise(seq_list):
    # Token -> base mapping from 02b_embed_center.py spans() (slow-tokenizer branch):
    # special tokens are found by id, the remaining tokens must spell the sequence exactly,
    # and their cumulative lengths give each token's [start, end) bases.
    # Previous token = the adjacent token before the centre one, as in 12_repool_center.py.
    enc = tok(seq_list, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    ids, am = enc["input_ids"], enc["attention_mask"]
    ids_np, am_np = ids.numpy(), am.numpy().astype(bool)
    c_idx = np.empty(len(seq_list), dtype=np.int64)
    p_idx = np.empty(len(seq_list), dtype=np.int64)
    seen = collections.Counter()
    for i, seq in enumerate(seq_list):
        body = am_np[i] & ~np.isin(ids_np[i], special_ids)
        pieces = tok.convert_ids_to_tokens(ids_np[i][body].tolist())
        lens = np.array([len(p) for p in pieces])
        assert int(lens.sum()) == len(seq), \
            f"reconstructed length {int(lens.sum())} != sequence length {len(seq)}"
        assert "".join(pieces) == seq, "reconstructed tokens do not spell the sequence"
        end = np.cumsum(lens)
        start = end - lens
        pos = np.flatnonzero(body)                     # input positions of the real tokens
        c = np.flatnonzero((start <= CENTER_BASE) & (end > CENTER_BASE))
        assert len(c) == 1, "expected exactly one token covering base 300"
        c = int(c[0])
        assert c >= 1, "no token before the centre token"
        assert end[c - 1] == start[c], "previous token is not adjacent"
        c_idx[i], p_idx[i] = pos[c], pos[c - 1]
        seen[(int(pos[c]), int(start[c]), int(end[c]) - 1,
              int(pos[c - 1]), int(start[c - 1]), int(end[c - 1]) - 1)] += 1
    return {"ids": ids, "am": am, "c": torch.from_numpy(c_idx), "p": torch.from_numpy(p_idx)}, seen


t0 = time.time()
toks, seen = {}, collections.Counter()
for s in ("train", "test"):
    toks[s], n = tokenise(seqs[s])
    seen += n
print(f"tokenised {sum(len(v) for v in seqs.values())} sequences in {time.time() - t0:.1f}s; "
      f"input shape {tuple(toks['train']['ids'].shape)}")

cls_first = bool(all((toks[s]["ids"][:, 0] == tok.cls_token_id).all() for s in toks))
print(f"<cls> (id {tok.cls_token_id}) is the first input token of every row: {cls_first}; "
      f"eos token: {tok.eos_token}")
for (ci, cs, ce, pi, ps, pe), n in seen.most_common():
    print(f"{n:6d} rows: centre token = input index {ci} (bases {cs}-{ce}), "
          f"previous token = input index {pi} (bases {ps}-{pe})")

# One row spelled out, to read by eye
r0 = toks["train"]
ci, pi = int(r0["c"][0]), int(r0["p"][0])
row_tokens = tok.convert_ids_to_tokens(r0["ids"][0].tolist())
print(f"row 0: input[0] = {row_tokens[0]!r}; input[{pi}] = {row_tokens[pi]!r}, input[{ci}] = {row_tokens[ci]!r}; "
      f"sequence[294:300] = {seqs['train'][0][294:300]!r}, sequence[300:306] = {seqs['train'][0][300:306]!r}")
CENTRE_INFO = [{"rows": n, "centre_input_index": ci, "centre_bases": [cs, ce],
                "previous_input_index": pi, "previous_bases": [ps, pe]}
               for (ci, cs, ce, pi, ps, pe), n in seen.most_common()]

## Model, pooling, metrics

In [ ]:
def load_model(repo):
    # try_dna_models.py load_model(): NT-v2 registers AutoModelForMaskedLM but not AutoModel,
    # so both are tried.
    from transformers import AutoModel, AutoModelForMaskedLM

    errors = []
    for cls in (AutoModel, AutoModelForMaskedLM):
        try:
            return cls.from_pretrained(repo, revision=MODEL_REVISION, trust_remote_code=True), cls.__name__
        except Exception as exc:
            errors.append(f"{cls.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError(" | ".join(errors))


def token_states(model, input_ids, attention_mask):
    # try_dna_models.py get_hidden(): last-layer vector of every token. Same logic, minus its
    # torch.no_grad() wrapper, because Run B needs gradients (Run A adds no_grad itself).
    out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state
    if getattr(out, "hidden_states", None):
        return out.hidden_states[-1]
    if isinstance(out, (tuple, list)):
        return out[0]
    raise RuntimeError("could not find per-token vectors in the model output")


def pooled(model, t, rows):
    # center_plus_prev_concat: [centre token ++ previous token], in that order
    # (12_repool_center.py: np.concatenate([cen, prev], axis=1)).
    h = token_states(model, t["ids"][rows].to(DEVICE), t["am"][rows].to(DEVICE))
    r = torch.arange(h.shape[0], device=h.device)
    return torch.cat([h[r, t["c"][rows].to(DEVICE)], h[r, t["p"][rows].to(DEVICE)]], dim=-1)


def make_head(in_dim):
    # 03_train_head.py: input -> 256 -> ReLU -> Dropout(0.2) -> classes
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, len(CLASSES)))


def evaluate(y_true, y_pred):
    # 03_train_head.py evaluate(): accuracy, macro-F1, per-class precision/recall/F1, confusion matrix
    from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                                 precision_recall_fscore_support)

    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0)), 4),
        "per_class": {LABEL_NAMES[c]: {"label": int(c), "precision": round(float(p[i]), 4),
                                       "recall": round(float(r[i]), 4), "f1": round(float(f[i]), 4),
                                       "support": int(s[i])} for i, c in enumerate(CLASSES)},
        "confusion_matrix": {"rows": "true label", "cols": "predicted label",
                             "names": [LABEL_NAMES[c] for c in CLASSES],
                             "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()},
    }


def print_report(title, m):
    print(f"\n--- {title} ---")
    print(f"test accuracy {m['accuracy']:.4f}   macro-F1 {m['macro_f1']:.4f}")
    for name, v in m["per_class"].items():
        print(f"{name:>10}  recall {v['recall']:.3f}  precision {v['precision']:.3f}  n {v['support']}")


t0 = time.time()
backbone, loaded_via = load_model(MODEL_REPO)
backbone.to(DEVICE)
HIDDEN = backbone.config.hidden_size
n_params = sum(p.numel() for p in backbone.parameters())
print(f"loaded {MODEL_REPO} via {loaded_via} in {time.time() - t0:.1f}s: "
      f"{n_params / 1e6:.1f}M parameters, hidden size {HIDDEN}, head input {2 * HIDDEN}")

## Run A: frozen backbone, head only

The backbone never changes here, so each sequence goes through it once and the head trains on the saved vectors, exactly as the local pipeline does (`02b_embed_center.py` + `12_repool_center.py` + `03_train_head.py`). Best epoch = lowest validation loss.

In [ ]:
def embed_frozen(t):
    backbone.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(t["ids"]), BATCH_SIZE):
            out.append(pooled(backbone, t, torch.arange(i, min(i + BATCH_SIZE, len(t["ids"])))).float().cpu().numpy())
    return np.concatenate(out)


t_a = time.time()
XA, embed_secs = {}, {}
for s in ("train", "test"):
    t0 = time.time()
    XA[s] = embed_frozen(toks[s])
    # phase1_common.py pair_features(): this variant was built from the float16 copy of the tokens
    XA[s] = XA[s].astype(np.float16).astype(np.float32)
    embed_secs[s] = time.time() - t0
    print(f"embedded {s}: {XA[s].shape} in {embed_secs[s]:.1f}s ({1000 * embed_secs[s] / len(XA[s]):.1f} ms/sequence)")

# From here on: 03_train_head.py (StandardScaler fit on the fit rows, then train_mlp()), on CPU.
from sklearn.preprocessing import StandardScaler

to_idx = {c: i for i, c in enumerate(CLASSES)}
yi = {s: np.array([to_idx[v] for v in y[s]]) for s in y}
scaler_a = StandardScaler().fit(XA["train"][fit_rows])


def train_mlp(X_fit, y_fit, X_val, y_val):
    # 03_train_head.py train_mlp(): Adam, early stopping on validation loss, best weights restored
    torch.manual_seed(HEAD_SEED)
    gen = torch.Generator().manual_seed(HEAD_SEED)
    Xf, yf = torch.tensor(X_fit, dtype=torch.float32), torch.tensor(y_fit, dtype=torch.long)
    Xv, yv = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)
    model = make_head(Xf.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=A_LR, weight_decay=A_WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best = {"val_loss": float("inf"), "epoch": 0, "state": None}
    history, since_best, t0 = [], 0, time.time()
    for epoch in range(1, A_MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(Xf), generator=gen)
        total = 0.0
        for i in range(0, len(Xf), A_BATCH):
            b = perm[i:i + A_BATCH]
            opt.zero_grad()
            loss = loss_fn(model(Xf[b]), yf[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            logits = model(Xv)
            val_loss = float(loss_fn(logits, yv))
            val_acc = float((logits.argmax(1) == yv).float().mean())
        history.append({"epoch": epoch, "train_loss": round(total / len(Xf), 5),
                        "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 4),
                        "elapsed_s": round(time.time() - t0, 2)})
        if val_loss < best["val_loss"] - 1e-4:
            best = {"val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
            since_best = 0
        else:
            since_best += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"  epoch {epoch:3d}  train loss {total / len(Xf):.4f}  val loss {val_loss:.4f}  "
                  f"val acc {val_acc:.4f}  elapsed {time.time() - t0:.1f}s")
        if since_best >= A_PATIENCE:
            print(f"  early stop at epoch {epoch} (no val-loss improvement for {A_PATIENCE} epochs); "
                  f"restoring epoch {best['epoch']}  elapsed {time.time() - t0:.1f}s")
            break
    model.load_state_dict(best["state"])
    model.eval()
    return model, best, history


print(f"\nRun A head: fit on {len(fit_rows)}, best epoch picked on {len(val_rows)} validation rows")
t0 = time.time()
head_a, best_a, hist_a = train_mlp(scaler_a.transform(XA["train"][fit_rows]), yi["train"][fit_rows],
                                   scaler_a.transform(XA["train"][val_rows]), yi["train"][val_rows])
head_a_secs = time.time() - t0
with torch.no_grad():
    pred_a = np.array(CLASSES)[head_a(torch.tensor(scaler_a.transform(XA["test"]),
                                                   dtype=torch.float32)).argmax(1).numpy()]
run_a = evaluate(y["test"], pred_a)
run_a["training"] = {
    "backbone": "frozen (eval mode, no gradients); each sequence embedded once",
    "features": "center_plus_prev_concat, rounded through float16, StandardScaler fit on the fit rows",
    "optimizer": f"Adam(lr={A_LR}, weight_decay={A_WEIGHT_DECAY})", "batch_size": A_BATCH,
    "max_epochs": A_MAX_EPOCHS, "patience": A_PATIENCE, "selection": "lowest validation loss",
    "best_epoch": best_a["epoch"], "epochs_run": len(hist_a),
    "best_val_loss": round(best_a["val_loss"], 5), "best_val_acc": round(best_a["val_acc"], 4),
    "history": hist_a,
}
run_a["seconds"] = {"embed_train": round(embed_secs["train"], 1), "embed_test": round(embed_secs["test"], 1),
                    "embed_ms_per_sequence": round(1000 * sum(embed_secs.values()) / sum(len(v) for v in XA.values()), 2),
                    "head_training": round(head_a_secs, 1), "total": round(time.time() - t_a, 1)}
print_report(f"Run A: frozen NT-v2-50M + head (n={len(y['test'])})", run_a)
print(f"Run A total: {run_a['seconds']['total']:.1f}s")

## Run B: full fine-tune

Same head, same fit / validation rows, but the backbone trains too (AdamW, backbone lr 2e-5, head lr 1e-3, weight decay 0.01, batch 16). Best epoch = lowest validation loss, as in Run A; the test set is scored once, with that epoch's weights. No input scaler here: the head reads the backbone's vectors directly and both learn together.

In [ ]:
torch.manual_seed(HEAD_SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(HEAD_SEED)
head_b = make_head(2 * HIDDEN).to(DEVICE)
opt = torch.optim.AdamW([{"params": backbone.parameters(), "lr": LR_BACKBONE},
                         {"params": head_b.parameters(), "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
grad_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
loss_fn = nn.CrossEntropyLoss()
y_t = {s: torch.tensor(yi[s], dtype=torch.long) for s in yi}


def predict_logits(t, rows):
    backbone.eval()
    head_b.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), 4 * BATCH_SIZE):
            b = torch.as_tensor(rows[i:i + 4 * BATCH_SIZE])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out.append(head_b(pooled(backbone, t, b)).float().cpu())
    return torch.cat(out)


fit_t = torch.as_tensor(fit_rows)
n_steps = (len(fit_t) + BATCH_SIZE - 1) // BATCH_SIZE
gen = torch.Generator().manual_seed(HEAD_SEED)
best_b = {"val_loss": float("inf"), "epoch": 0, "state": None}
hist_b = []
t_b = time.time()
print(f"Run B: {len(fit_t)} fit rows, {n_steps} steps/epoch, {FT_EPOCHS} epoch(s), fp16 {USE_AMP}")
for epoch in range(1, FT_EPOCHS + 1):
    t0 = time.time()
    backbone.train()
    head_b.train()
    perm = fit_t[torch.randperm(len(fit_t), generator=gen)]
    total = 0.0
    for step, i in enumerate(range(0, len(perm), BATCH_SIZE), start=1):
        b = perm[i:i + BATCH_SIZE]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = head_b(pooled(backbone, toks["train"], b))
        loss = loss_fn(logits.float(), y_t["train"][b].to(DEVICE))
        if not torch.isfinite(loss):
            raise RuntimeError(f"loss is {loss.item()} at epoch {epoch} step {step}. "
                               "If fp16 is on, set USE_AMP = False in the device cell and rerun.")
        opt.zero_grad(set_to_none=True)
        grad_scaler.scale(loss).backward()
        grad_scaler.step(opt)
        grad_scaler.update()
        total += loss.item() * len(b)
        if step % max(1, n_steps // 10) == 0 or step == n_steps:
            print(f"  epoch {epoch} step {step:4d}/{n_steps}  loss {loss.item():.4f}  "
                  f"{time.time() - t0:.1f}s", flush=True)
    train_secs = time.time() - t0

    t1 = time.time()
    val_logits = predict_logits(toks["train"], val_rows)
    val_loss = float(loss_fn(val_logits, y_t["train"][val_rows]))
    val_acc = float((val_logits.argmax(1) == y_t["train"][val_rows]).float().mean())
    val_secs = time.time() - t1
    hist_b.append({"epoch": epoch, "train_loss": round(total / len(fit_t), 5),
                   "val_loss": round(val_loss, 5), "val_acc": round(val_acc, 4),
                   "train_s": round(train_secs, 1), "val_s": round(val_secs, 1),
                   "train_s_per_step": round(train_secs / n_steps, 4)})
    print(f"epoch {epoch}/{FT_EPOCHS}  train loss {total / len(fit_t):.4f}  val loss {val_loss:.4f}  "
          f"val acc {val_acc:.4f}  |  elapsed: train {train_secs:.1f}s ({train_secs / n_steps:.3f} s/step), "
          f"validation {val_secs:.1f}s", flush=True)
    if val_loss < best_b["val_loss"]:
        best_b = {"val_loss": val_loss, "val_acc": val_acc, "epoch": epoch,
                  "state": ({k: v.detach().clone() for k, v in backbone.state_dict().items()},
                            {k: v.detach().clone() for k, v in head_b.state_dict().items()})}

print(f"best epoch on validation: {best_b['epoch']} of {FT_EPOCHS}; scoring the test set with its weights")
backbone.load_state_dict(best_b["state"][0])
head_b.load_state_dict(best_b["state"][1])
t1 = time.time()
pred_b = np.array(CLASSES)[predict_logits(toks["test"], np.arange(len(y["test"]))).argmax(1).numpy()]
test_secs = time.time() - t1
run_b = evaluate(y["test"], pred_b)
run_b["training"] = {
    "backbone": "fully fine-tuned", "features": "center_plus_prev_concat, straight into the head",
    "optimizer": f"AdamW(backbone lr={LR_BACKBONE}, head lr={LR_HEAD}, weight_decay={WEIGHT_DECAY})",
    "batch_size": BATCH_SIZE, "epochs": FT_EPOCHS, "steps_per_epoch": n_steps, "fp16_autocast": USE_AMP,
    "selection": "lowest validation loss", "best_epoch": best_b["epoch"],
    "best_val_loss": round(best_b["val_loss"], 5), "best_val_acc": round(best_b["val_acc"], 4),
    "history": hist_b,
}
run_b["seconds"] = {"test_eval": round(test_secs, 1), "total": round(time.time() - t_b, 1)}
print_report(f"Run B: fine-tuned NT-v2-50M + head (n={len(y['test'])})", run_b)
print(f"Run B total: {run_b['seconds']['total']:.1f}s")

## Before / after

In [ ]:
results = {
    "smoke": SMOKE,
    "note": ("SMOKE run: tiny sample, 1 epoch. These numbers only show the notebook runs."
             if SMOKE else "Full run."),
    "model": MODEL_REPO, "model_revision": MODEL_REVISION,
    "dataset": f"{DATASET_REPO} :: {DATASET_CONFIG}", "dataset_revision": DATASET_REVISION,
    "device": str(DEVICE), "gpu": GPU_NAME, "fp16_autocast": USE_AMP,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__},
    "split": {"train_chromosomes": "chr1-19, X, Y", "test_chromosomes": "chr20, chr21",
              "sampling_seed": SEED, "n_train": int(len(y["train"])), "n_fit": int(len(fit_rows)),
              "n_val": int(len(val_rows)), "n_test": int(len(y["test"])),
              "val_split_seed": HEAD_SEED,
              "same_rows_as_local_pipeline": SAME_SAMPLE_AS_LOCAL},
    "centre_token": {"cls_is_first_input_token": cls_first, "found": CENTRE_INFO},
    "head": f"Linear({2 * HIDDEN},256) -> ReLU -> Dropout(0.2) -> Linear(256,{len(CLASSES)})",
    "run_A_frozen": run_a,
    "run_B_finetuned": run_b,
    "accuracy_gain_B_minus_A": round(run_b["accuracy"] - run_a["accuracy"], 4),
    "total_seconds": round(time.time() - T_START, 1),
}
with open("results.json", "w") as fh:
    json.dump(results, fh, indent=2)
    fh.write("\n")

if SMOKE:
    print("SMOKE RUN: tiny sample, 1 epoch. Not a result. Set SMOKE = False for the real numbers.\n")
names = [LABEL_NAMES[c] for c in CLASSES]
print(f"{'':<22}{'accuracy':>9}" + "".join(f"{'recall ' + n:>17}" for n in names))
for title, m in (("A  frozen", run_a), ("B  fine-tuned", run_b)):
    print(f"{title:<22}{m['accuracy']:>9.4f}" + "".join(f"{m['per_class'][n]['recall']:>17.4f}" for n in names))
print(f"{'B - A':<22}{results['accuracy_gain_B_minus_A']:>+9.4f}" + "".join(
    f"{run_b['per_class'][n]['recall'] - run_a['per_class'][n]['recall']:>+17.4f}" for n in names))
print(f"\ntest rows: {len(y['test'])}   device: {DEVICE}   total time: {results['total_seconds']:.0f}s")
print("wrote results.json")